# Actividad 2 · Evaluar un modelo de clasificación

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)  
**Carácter:** obligatoria.

> Realizá solamente uno de los dos recorridos equivalentes: **SQL + Python** o **SQL + R**.

En la Unidad N° 5 construiste un primer modelo de clasificación para anticipar cancelaciones. En esta actividad no vas a entrenar un modelo nuevo: vas a trabajar con **predicciones ya generadas sobre un conjunto de prueba** para aprender a evaluar qué acierta, dónde se equivoca y qué información aporta cada métrica.

El caso utiliza nuevamente una variable binaria:

- `is_canceled = 1`: la reserva se canceló.
- `is_canceled = 0`: la reserva no se canceló.
- `predicted_canceled`: predicción generada por el modelo.

El notebook es **autocontenido** y genera internamente los datos necesarios.

## 1. Preparar los datos de evaluación

Ejecutá la siguiente celda. Se crearán 100 observaciones de prueba con valores reales y predicciones ya generadas por un clasificador.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Construimos un conjunto de prueba de 100 reservas con esta distribución:
# TN = 72, FP = 8, FN = 6, TP = 14
reales = (
    [0] * 72 +  # verdaderos negativos
    [0] * 8  +  # falsos positivos
    [1] * 6  +  # falsos negativos
    [1] * 14    # verdaderos positivos
)

predichos = (
    [0] * 72 +
    [1] * 8  +
    [0] * 6  +
    [1] * 14
)

evaluacion = pd.DataFrame({
    "id_reserva": range(1, 101),
    "is_canceled": reales,
    "predicted_canceled": predichos
})

# Mezclamos el orden para que el caso se vea como un conjunto de observaciones real.
evaluacion = evaluacion.sample(frac=1, random_state=13).reset_index(drop=True)

con = sqlite3.connect(":memory:")
evaluacion.to_sql("evaluacion_modelo", con, index=False, if_exists="replace")

print("Datos preparados:", len(evaluacion), "reservas del conjunto de prueba.")
evaluacion.head(10)

## 2. Comparar valores reales y predichos con SQL

Antes de calcular métricas, organizá los resultados según las cuatro combinaciones posibles entre el valor real y la predicción.

In [ ]:
consulta = """
SELECT
    is_canceled,
    predicted_canceled,
    COUNT(*) AS cantidad
FROM evaluacion_modelo
GROUP BY is_canceled, predicted_canceled
ORDER BY is_canceled, predicted_canceled
"""

resumen = pd.read_sql_query(consulta, con)
resumen

## 3. Interpretar la matriz de confusión

Si consideramos como **positivo** el evento “la reserva se cancela”, las cuatro situaciones son:

- **Verdadero positivo (TP):** el modelo predijo cancelación y la reserva efectivamente se canceló.
- **Verdadero negativo (TN):** predijo que no se cancelaría y efectivamente no se canceló.
- **Falso positivo (FP):** predijo cancelación, pero la reserva no se canceló.
- **Falso negativo (FN):** predijo que no se cancelaría, pero finalmente se canceló.

En un problema real, FP y FN no necesariamente tienen el mismo costo.

In [ ]:
tn = int(((evaluacion["is_canceled"] == 0) & (evaluacion["predicted_canceled"] == 0)).sum())
fp = int(((evaluacion["is_canceled"] == 0) & (evaluacion["predicted_canceled"] == 1)).sum())
fn = int(((evaluacion["is_canceled"] == 1) & (evaluacion["predicted_canceled"] == 0)).sum())
tp = int(((evaluacion["is_canceled"] == 1) & (evaluacion["predicted_canceled"] == 1)).sum())

matriz = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Real: no cancela (0)", "Real: cancela (1)"],
    columns=["Predice: no cancela (0)", "Predice: cancela (1)"]
)

matriz

## 4. Accuracy: ¿qué proporción total de casos clasificó correctamente?

\[
Accuracy=\frac{TP+TN}{TP+TN+FP+FN}
\]

Es una medida global fácil de interpretar, pero puede resultar insuficiente si las clases están desbalanceadas o si ciertos errores son más importantes que otros.

In [ ]:
accuracy = (tp + tn) / (tp + tn + fp + fn)
print(f"Accuracy = {accuracy:.3f}  ({accuracy*100:.1f}%)")

En este caso, una accuracy alta puede dar una primera impresión favorable. Pero todavía necesitamos saber **cómo se comporta el modelo específicamente frente a las cancelaciones**.

## 5. Precision: cuando el modelo predice cancelación, ¿cuántas veces acierta?

\[
Precision=\frac{TP}{TP+FP}
\]

Una precision alta significa que las alertas de cancelación generadas por el modelo suelen ser correctas.

In [ ]:
precision = tp / (tp + fp)
print(f"Precision = {precision:.3f}  ({precision*100:.1f}%)")

## 6. Recall: de todas las cancelaciones reales, ¿cuántas logra detectar?

\[
Recall=\frac{TP}{TP+FN}
\]

Un recall alto significa que el modelo deja escapar pocas cancelaciones reales.

In [ ]:
recall = tp / (tp + fn)
print(f"Recall = {recall:.3f}  ({recall*100:.1f}%)")

## 7. F1-score: combinar precision y recall

\[
F1=2\cdot\frac{Precision\cdot Recall}{Precision+Recall}
\]

F1 resume el equilibrio entre precision y recall. Es útil cuando ambas dimensiones importan y no queremos evaluar el modelo solamente por la proporción total de aciertos.

In [ ]:
f1 = 2 * precision * recall / (precision + recall)
print(f"F1-score = {f1:.3f}  ({f1*100:.1f}%)")

## 8. Leer las métricas en contexto

Con los resultados obtenidos:

- **Accuracy** responde: ¿qué proporción total de reservas clasificó correctamente?
- **Precision** responde: cuando el modelo alerta una cancelación, ¿qué tan confiable es esa alerta?
- **Recall** responde: de todas las cancelaciones que realmente ocurren, ¿cuántas detecta?
- **F1-score** ayuda a resumir el equilibrio entre precision y recall.

Ahora pensá en las consecuencias:

- Un **falso positivo** puede llevar al hotel a intervenir sobre una reserva que finalmente no iba a cancelarse.
- Un **falso negativo** implica no anticipar una cancelación que efectivamente ocurrirá.

Por eso, **no existe una única métrica “mejor” para todos los problemas**. La elección depende de qué error resulte más costoso en el contexto analizado.

## 9. Cierre de la actividad

Al finalizar, deberías poder:

- interpretar una **matriz de confusión**;
- diferenciar TP, TN, FP y FN;
- explicar qué informa **accuracy**;
- explicar qué informa **precision**;
- explicar qué informa **recall**;
- comprender para qué sirve **F1-score**;
- justificar por qué la evaluación de un modelo no debería reducirse a mirar un único número.

En la **Práctica supervisada** vas a aplicar los contenidos de regresión lineal simple. Más adelante, en la **Entrega 6 del TPI**, vas a utilizar estas ideas para evaluar el modelo de clasificación construido sobre el problema de cancelación de reservas.

In [ ]:
con.close()